In [ ]:
import datetime
from polars.exceptions import ColumnNotFoundError
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.cm import ScalarMappable
from matplotlib.colors import (
    BoundaryNorm,
    LinearSegmentedColormap,
    hsv_to_rgb,
    rgb_to_hsv,
)
from functools import partial
from pathlib import Path
import numpy as np
import xarray as xr
import polars as pl
from scipy.ndimage import gaussian_filter
from scipy.interpolate import splev, splprep
import colormaps
from tqdm import tqdm, trange
import dask.array as darr
from dask.array.lib.stride_tricks import sliding_window_view
from jetutils.definitions import (
    RESULTS,
    RADIUS,
    DATADIR,
    FIGURES,
    MONTH_NAMES,
    N_WORKERS,
    do_rle,
    polars_to_xarray,
    xarray_to_polars,
    Timer,
    get_index_columns,
    get_region,
    explode_rle,
    compute,
)
from jetutils.plots import Clusterplot, make_transparent, COLORS_EXT, COLORS, num2tex, plot_seasonal, plot_dayofyear_trends
from jetutils.data import extract, standardize, standardize_polars_dtypes
from jetutils.geospatial import (
    detect_contours_lonlat,
    haversine,
    compute_alignment,
    sort_by_index_then_newindex,
    diff_exp,
    sort_by_index_then_difflon,
    gather_normal_da_jets,
    interp_jets_to_zero_one,
    detect_contours,
    create_jet_relative_dataset
)
from jetutils.derived_quantities import compute_norm_derivative, find_axis
from jetutils.jet_finding import (
    coarsen_da,
    preprocess_ds,
    average_jet_categories,
    jet_integral_haversine,
    jet_position_as_da,
    extract_features,
    has_periodic_x,
    round_polars,
    do_everything, 
    gaussian_smooth_func,
    online_bias_correct,
    join_on_ds,
    find_all_jets,
    track_jets,
    pers_from_cross,
    persistence_expr,
    spells_from_cross,
    spells_from_cross_catd,
    spells_from_cross_catd_simple,
    connected_from_cross,
    compute_jet_props,
    to_one_large,
    categorize_jets,
)

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline

In [ ]:
standardize(xr.open_dataset(f"{DATADIR}/thetalev/PV_19590219_06.nc"))["PV"][0, 3].plot()

# download cesm nice script, most up to date as of june 2026

In [ ]:
import time
from itertools import pairwise, product
from pathlib import Path
from jetutils.definitions import DATADIR
import numpy as np
import xarray as xr
from tqdm import tqdm
import socket
from urllib.request import urlretrieve
from urllib.error import HTTPError, ContentTooShortError, URLError

socket.setdefaulttimeout(180)


## functions:


def get_url(
    varname: str,
    experiment: str,
    member: str,
    timebounds: str,
    minlon: float,
    maxlon: float,
    minlat: float,
    maxlat: float,
    year: int,
):
    h = 6 if varname in ["U", "V", "T", "Q", "OMEGA", "Z3"] else 1  # and sometimes 7??

    base_url = f"https://tds.gdex.ucar.edu/thredds/ncss/grid/files/d651056/CESM2-LE/atm/proc/tseries/day_1/{varname}/b.e21.{experiment}.f09_g17.LE2-{member}.cam.h{h}.{varname}.{timebounds}.nc"
    specs = f"var={varname}&north={maxlat}&west={minlon}&east={maxlon}&south={minlat}&horizStride=1&time_start={year}-01-01T00:00:00Z&time_end={year}-12-31T00:00:00Z&&&accept=netcdf4ext"
    return f"{base_url}?{specs}"


class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)


def _download_url(url, output_path):
    with DownloadProgressBar(
        unit="B", unit_scale=True, miniters=1, desc=url.split("/")[-1]
    ) as t:
        urlretrieve(url, filename=output_path, reporthook=t.update_to)


def download_url(url, output_path, retries=10):
    while retries > 0:
        try:
            _download_url(url, output_path)
            return
        except (HTTPError, ContentTooShortError, URLError, TimeoutError):
            retries = retries - 1
            time.sleep(1.0)
            continue
    _download_url(url, output_path)


## constants:
FORCING_VARIANTS = ["smbb", "cmip6"]
EXPERIMENTS = {
    variant: {
        "historical": f"BHIST{variant}",
        "ssp370": f"BSSP370{variant}",
    }
    for variant in FORCING_VARIANTS
}
YEARBOUNDS = {
    "historical": np.arange(1850, 2021, 10),
    "ssp370": np.arange(2015, 2106, 10),
}
YEARBOUNDS["historical"][-1] = YEARBOUNDS["historical"][-1] - 5
YEARBOUNDS["ssp370"][-1] = YEARBOUNDS["ssp370"][-1] - 4
TIMEBOUNDS = {
    key: [f"{year1}0101-{year2 - 1}1231" for year1, year2 in pairwise(val)]
    for key, val in YEARBOUNDS.items()
}
YEARS = {
    key: [
        list(range(max(year1, 2060) if key == "future" else year1, year2))
        for year1, year2 in pairwise(val)
    ]
    for key, val in YEARBOUNDS.items()
}

MEMBERS1 = [
    f"{year}.{str(number).zfill(3)}"
    for year, number in zip(range(1001, 1201, 20), range(1, 11))
]
for startyear in [1231, 1251, 1281, 1301]:
    MEMBERS1.extend(f"{startyear}.{str(number).zfill(3)}" for number in range(1, 11))

MEMBERS2 = [
    f"r{number}i{year}p1f1" for year, number in zip(range(1001, 1201, 20), range(1, 11))
]
for startyear in [1231, 1251, 1281, 1301]:
    MEMBERS2.extend(f"r{number}i{startyear}p1f1" for number in range(1, 11))


## config:
variables = ["U", "V", "T"]  # your responsibility to make sure it exists
forcing_variant = "cmip6"  # or smbb
assert forcing_variant in FORCING_VARIANTS
minlon = -80
maxlon = 40
minlat = 15
maxlat = 80

years = {
    "historical": np.arange(1960, 2011),
    "ssp370": np.arange(2015, 2100),
}

periods_to_do = ["historical", "ssp370"]
members_to_do = list(range(50))
# levels = "all" # cannot specify multiple levels with netcdf-select, only one or all
basepath = Path(f"{DATADIR}/CESM2/high_wind")

## combine into arguments
timebounds_to_do = {}
years_split = {}
for key, val in TIMEBOUNDS.items():
    these_years = YEARS[key]
    years_to_dl = years[key]
    grr = np.unique(
        [i for i, y in enumerate(these_years) if np.isin(years_to_dl, y).any()]
    )
    timebounds_to_do[key] = [TIMEBOUNDS[key][i] for i in grr]
    years_split[key] = [np.intersect1d(these_years[i], years_to_dl) for i in grr]

experiment_dict = EXPERIMENTS[forcing_variant]
members1 = np.array(MEMBERS1)[members_to_do]
members2 = np.array(MEMBERS2)[members_to_do]
iterator = product(periods_to_do, variables, zip(members1, members2))

for period, varname, (member1, member2) in iterator:
    scratchdir = basepath.joinpath(period).joinpath("raw")
    scratchdir.mkdir(exist_ok=True, parents=True)
    experiment = EXPERIMENTS[forcing_variant][period]
    for years_, tb in zip(years_split[period], timebounds_to_do[period]):
        for year in years_:
            url = get_url(
                varname, experiment, member1, tb, minlon, maxlon, minlat, maxlat, year
            )
            scratchpath = scratchdir.joinpath(f"{varname}-{member2}-{year}.nc")
            try:
                xr.open_dataset(scratchpath)
            except (FileNotFoundError, ValueError):
                pass
            else:
                continue
            download_url(url, scratchpath)
            break
        break
    break

In [ ]:
xr.open_dataset(scratchpath)

# Altair tests

In [ ]:
all_jets_one_df = pl.read_parquet(
    "/Users/bandelol/Documents/code_local/data/exp6/all_jets_one_df.parquet"
).cast({"time": pl.Datetime("ms")})
props_uncat = pl.read_parquet(
    "/Users/bandelol/Documents/code_local/data/exp6/props_as_df_uncat.parquet"
).cast({"time": pl.Datetime("ms")})
ds_test = open_dataset("/Users/bandelol/Documents/code_local/data/flat_wind_sample.nc")
df_test = xarray_to_polars(
    ds_test.isel(time=0)["u"].sel(lon=slice(-80, 40, 1), lat=slice(20, 80, 1))
).drop("time")
alt.data_transformers.disable_max_rows()

In [ ]:
import altair as alt
from vega_datasets import data

source = alt.topo_feature(data.world_110m.url, "countries")

input_dropdown = alt.binding_select(
    options=[
        "albers",
        "albersUsa",
        "azimuthalEqualArea",
        "azimuthalEquidistant",
        "conicEqualArea",
        "conicEquidistant",
        "equalEarth",
        "equirectangular",
        "gnomonic",
        "mercator",
        "naturalEarth1",
        "orthographic",
        "stereographic",
        "transverseMercator",
    ],
    name="Projection ",
)

param_projection = alt.param(value="equalEarth", bind=input_dropdown)

# countries = alt.Chart(source, width=500, height=300).mark_geoshape(
#     fill='lightgray',
#     stroke='gray'
# ).project(
#     type=alt.expr(param_projection.name)
# ).add_params(param_projection)

wind = (
    alt.Chart(df_test, width=500, height=300)
    .mark_rect()
    .encode(
        alt.X("lon:O"),
        alt.Y("lat:O"),
        alt.Color("u:Q"),
        longitude=alt.Longitude("lon:Q"),
        latitude=alt.Latitude("lat:Q"),
    )
    .project(type=alt.expr(param_projection.name))
    .add_params(param_projection)
)

wind
# No channel encoding options are specified in this chart
# so the code is the same as for the method-based syntax.

# presentation plots

## som

In [ ]:
props_uncat = exp.props_as_df(False).cast({"time": pl.Datetime("ms")})
props_uncat = pl.concat(
    [
        props_uncat.filter(
            pl.col("is_polar").mean().over("time", "jet ID") > 0.5,
            pl.col("int") > 1.5e9,
        ),
        props_uncat.filter(pl.col("is_polar").mean().over("time", "jet ID") <= 0.5),
    ]
).sort("time", "jet ID")
all_jets_one_df = props_uncat[["time", "jet ID"]].join(
    all_jets_one_df, on=["time", "jet ID"]
)
props_uncat = props_uncat.with_columns(
    **{"jet ID": pl.col("jet ID").rle_id().over("time")}
)
all_jets_one_df = all_jets_one_df.with_columns(
    **{"jet ID": pl.col("jet ID").rle_id().over("time")}
)
jet_pos_da = jet_position_as_da(all_jets_one_df)
props_as_df = average_jet_categories(props_uncat)

In [ ]:
from jetutils.plots import COLORS
from matplotlib.colors import rgb2hex

In [ ]:
[rgb2hex(color) for color in COLORS_EXT]

In [ ]:
ds_center = xr.open_dataset(
    "/Users/bandelol/Documents/code_local/data/exp6/uvs_som_4_4_200_pbc_center.nc"
)

In [ ]:
ds_center = ds_center.load()
df_center = pl.from_pandas(ds_center.to_dataframe().reset_index())
centers_all_jets = find_all_jets(df_center, base_s_thresh=18, alignment_thresh=0.5)
centers_all_jets = is_polar_gmix(
    centers_all_jets, ("lon", "lat", "lev"), n_components=2, n_init=40
)

new_id = []
for clu, jets in centers_all_jets.group_by("cluster", maintain_order=True):
    njets = jets["jet ID"].n_unique()
    for idx, jet in jets.group_by("jet ID", maintain_order=True):
        new_jet_id = jet.with_columns(
            pl.col("jet ID")
            + (pl.col("is_polar") > 0.5)
            .cast(pl.UInt16)
            .diff()
            .abs()
            .fill_null(0)
            .cum_sum()
            .cast(pl.UInt32)
            * njets
        )
        njets = jets["jet ID"].n_unique() - 1 + new_jet_id["jet ID"].n_unique()
        new_id.append(new_jet_id.sort("lon"))
centers_all_jets = pl.concat(new_id)

centers_props_uncat = compute_jet_props(centers_all_jets)
centers_props = categorize_df_jets(centers_props_uncat, polar_cutoff=0.001)

In [ ]:
clu = Clusterplot(
    4, 4, get_region(ds_center), honeycomb=True, numbering=True, row_height=3
)
_ = clu.add_contourf(
    ds_center["s"],
    cmap=colormaps.gothic_r,
    # transparify=1,
    levels=[0, 8, 16, 24, 32, 40],
    cbar_kwargs={"label": "Wind speed [m/s]", "pad": 0.01},
)
# for indexer, jet in centers_all_jets.group_by(["cluster", "jet ID"], maintain_order=True):
#     ax = clu.axes[indexer[0]]
#     is_polar = jet["is_polar"].mean() >= 0.35
#     ax.plot(*jet[["lon", "lat"]].to_numpy().T, lw=4, ls="solid")
clu.resize_relative([0.95, 1])
clu.fig.savefig("/Users/bandelol/Documents/code_local/local_figs/som_wind.png")

In [ ]:
clu = Clusterplot(1, 1, get_region(ds_center), row_height=7)
clu.fig.savefig("/Users/bandelol/Documents/code_local/local_figs/region.png")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from simpsom_dask import plots as splots
from simpsom_dask.neighborhoods import Neighborhoods
import colormaps
import numpy as np
from jetutils.plots import TEXTWIDTH_IN

outer_grid, inner_grid, coords, outermask = splots.create_outer_grid(4, 4)
nei = Neighborhoods(4, 4, "hexagons", PBC=True, dist_type="grid")
cmap = colormaps.cet_l_bmw_r
cmap = LinearSegmentedColormap.from_list(
    "hoho", cmap(np.linspace(0.0, 0.75, cmap.N // 2))
)
from_ = 5
distances = nei.distances[from_, outer_grid]
# distances = np.full(len(coords), np.nan)
edgecolors = np.full(len(coords), "black", dtype=object)
edgecolors[outermask] = "gray"
alphas = np.ones(len(coords))
alphas[outermask] = 0.7
fig, ax = splots.plot_map(
    coords,
    distances,
    None,
    "hexagons",
    draw_cbar=False,
    figsize=(0.5 * TEXTWIDTH_IN, 1.55),
    show=False,
    edgecolors=edgecolors,
    cmap=cmap,
    alphas=alphas,
    linewidths=1.5,
)
xlims = [
    np.amin(coords[~outermask][:, 0]) - 0.8,
    np.amax(coords[~outermask][:, 0]) + 0.8,
]
ylims = [np.amin(coords[~outermask][:, 1]) - 1, np.amax(coords[~outermask][:, 1]) + 1]
for i, c in enumerate(coords):
    x, y = c
    textcolor = "white" if distances[i] > 2 else "black"
    fontsize = 7 if outermask[i] else 12
    if x > xlims[0] and x < xlims[-1] and y > ylims[0] and y < ylims[-1]:
        ax.text(
            x,
            y,
            f"${outer_grid.flatten()[i] + 1}$",
            va="center",
            ha="center",
            color=textcolor,
            fontsize=fontsize,
        )
ax.set_xlim(xlims)
ax.set_ylim(ylims)
ax.set_aspect("equal")
# ax.set_title(f"Distance to cluster {from_ + 1}", pad=5)
plt.savefig(f"/Users/bandelol/Documents/code_local/local_figs/demo_dist.png")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from simpsom_dask import plots as splots
from simpsom_dask.neighborhoods import Neighborhoods
import colormaps
import numpy as np
from jetutils.plots import TEXTWIDTH_IN
from jetutils.definitions import FIGURES

nei = Neighborhoods(4, 4, "hexagons", PBC=False, dist_type="grid")
coords = nei.coordinates
cmap = colormaps.cet_l_bmw_r
cmap = LinearSegmentedColormap.from_list(
    "hoho", cmap(np.linspace(0.0, 0.75, cmap.N // 2))
)
from_ = 5
distances = nei.distances[from_, np.arange(16)]
# distances = np.full(len(coords), np.nan)
edgecolors = np.full(len(coords), "black", dtype=object)
alphas = np.ones(len(coords))
fig, ax = splots.plot_map(
    coords,
    distances,
    None,
    "hexagons",
    draw_cbar=False,
    figsize=(0.5 * TEXTWIDTH_IN, 1.55),
    show=False,
    edgecolors=edgecolors,
    cmap=cmap,
    linewidths=1.5,
)
fontsize = 10
for i, c in enumerate(coords):
    x, y = c
    textcolor = "white" if distances[i] > 1 else "black"
    ax.text(
        x,
        y,
        str(i),
        va="center",
        ha="center",
        color=textcolor,
        fontsize=fontsize,
        weight="bold",
    )
# ax.set_title(f"Distance to cluster {from_ + 1}", pad=5)
plt.savefig(f"{FIGURES}/demo_dist_no_pbc.png")

## jet detection

In [ ]:
wind_sample = xr.open_dataset(f"{DATADIR}/ERA5/sample_high.nc")
wind_sample = extract(
    wind_sample, minlon=-80, maxlon=40, minlat=15, maxlat=80
)
wind_sample["sigma"] = compute_norm_derivative(wind_sample, "s")
wind_sample_ = wind_sample.compute()
times = wind_sample["time"].values

In [ ]:
find_jets_kwargs = dict(
    n_coarsen=3,
    base_s_thresh=0.55,
    alignment_thresh=0.6,
    int_thresh_factor=0.6,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)
jets = find_all_jets(wind_sample, **find_jets_kwargs)
jets = categorize_jets(jets, feature_names=("s", "theta"))
props = compute_jet_props(jets)
phat_jets = to_one_large(jets)
cross_catd = track_jets(phat_jets)
cross = track_jets(jets)

In [ ]:
cross2, summary_comp = connected_from_cross(jets, cross, dist_thresh=1.2e6)

In [ ]:
spells_from_cross(jets, cross, n_STJ=30, n_EDJ=30)

In [ ]:
spells_from_cross_catd_simple(
    cross_catd,
    q_STJ=0.705,
    q_EDJ=0.836,
    minlen=datetime.timedelta(days=5),
    smooth=datetime.timedelta(hours=24),
    fill_holes=datetime.timedelta(hours=18),
)

In [ ]:
time_to_plot = np.random.choice(times)
ds = wind_sample_.sel(time=[time_to_plot]).copy()
thresholds: xr.DataArray | None = None
n_coarsen: int = 3
base_s_thresh: float = .6
alignment_thresh: float = 0.7
int_thresh_factor: float = 0.6
hole_size: int = 6
smooth_func = partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.7)

ds, ds_orig = preprocess_ds(ds.copy(), n_coarsen=n_coarsen, smooth_func=smooth_func)
df = xarray_to_polars(ds)
x_periodic = has_periodic_x(ds_orig)
index_columns = get_index_columns(
    df,
    (
        "member",
        "number",
        "forecast_init",
        "time",
        "cluster",
        "spell",
        "relative_index",
        "sample_index",
        "inside_index",
    ),
)

# thresholds
dl = np.radians(df["lon"].max() - df["lon"].min())
base_int_thresh = RADIUS * dl * base_s_thresh * np.cos(np.pi / 4) * int_thresh_factor
ic_nomemb = [ic for ic in index_columns if ic not in ["member", "number"]]
if base_s_thresh <= 1.0:
    thresholds = df.group_by(ic_nomemb).agg(
        pl.col("s").quantile(base_s_thresh).alias("s_thresh")
    )
    base_s_thresh = thresholds["s_thresh"].mean()  # disgusting
    base_int_thresh = (
        RADIUS * dl * base_s_thresh * np.cos(np.pi / 4) * int_thresh_factor
    )
elif thresholds is not None:
    thresholds = (
        xarray_to_polars(thresholds)
        .drop("quantile")
        .cast({"s": pl.Float32})
        .rename({"s": "s_thresh"})
    )
if thresholds is not None:
    df = df.join(thresholds, on=ic_nomemb)  # ugly
    df = df.with_columns(
        int_thresh=base_int_thresh * pl.col("s_thresh") / base_s_thresh
    )
    condition_expr = (pl.col("s") > pl.col("s_thresh")) & (
        pl.col("alignment") > alignment_thresh
    )
    condition_expr2 = pl.col("int") > pl.col("int_thresh")
else:
    condition_expr = (pl.col("s") > base_s_thresh) & (
        pl.col("alignment") > alignment_thresh
    )
    condition_expr2 = pl.col("int") > base_int_thresh

clu = Clusterplot(2, 2, get_region(wind_sample), row_height=5)
cmap1 = colormaps.greys
levels_wind = np.arange(0, 76, 15)
cmap1 = make_transparent(cmap1, len(levels_wind), 1, 1, 1)
norm1 = BoundaryNorm(levels_wind, cmap1.N, extend="max")
im1 = ScalarMappable(norm1, cmap1)
cmap2 = colormaps.viola
levels_sigma = MaxNLocator(12, symmetric=True).tick_values(-0.15, 0.15)
norm2 = BoundaryNorm(levels_sigma, cmap2.N, extend="neither")
im2 = ScalarMappable(norm2, cmap2)
cbar2 = clu.fig.colorbar(
    im2,
    ax=clu.axes,
    pad=0.01,
    shrink=0.95,
    spacing="proportional",
    label="Wind shear [$10^{-3}$/s]",
)
cbar2.ax.plot([0, 1], [0, 0], lw=3, color="black")
clu.fig.colorbar(
    im1,
    ax=clu.axes,
    pad=0.01,
    shrink=0.95,
    spacing="proportional",
    label="Wind speed [m/s]",
)
colors_jets = colormaps.bold(np.arange(20) % 10)

BLUEWHITERED = LinearSegmentedColormap.from_list(
    "bluewhitered",
    [
        colormaps.cet_l_bmw(0.34)[:3],
        COLORS_EXT[0],
        "#ffffff",
        COLORS_EXT[9],
        COLORS_EXT[11],
    ],
)

lon = ds["lon"].values
lat = ds["lat"].values
clu.axes[0].contourf(
    lon - clu.central_longitude, lat, ds["s"][0], cmap=cmap1, norm=norm1
)

clu.axes[1].contourf(
    lon - clu.central_longitude, lat, ds["sigma"][0] * 1000, cmap=cmap2, norm=norm2
)
clu.axes[1].contour(
    lon - clu.central_longitude,
    lat,
    ds["sigma"][0] * 1000,
    levels=[0.0],
    colors="black",
    linewidths=1,
)
for i in [2, 3]:
    clu.axes[i].contourf(
        lon - clu.central_longitude, lat, ds["s"][0], cmap=cmap1, norm=norm1
    )
# contours
repeat_lons = 120 if x_periodic else 0.0
all_contours = detect_contours_lonlat(
    (ds["sigma"] > 0).astype(np.uint8),
    [0.0],
    processes=N_WORKERS,
    repeat_lons=repeat_lons,
    do_round=False,
).drop("level")

int_expr = jet_integral_haversine(pl.col("lon"), pl.col("lat"), pl.col("s"))
int_expr = int_expr.over([*index_columns, "jet ID"])
distance_ends_expr = haversine(
    pl.col("lon").first(),
    pl.col("lat").first(),
    pl.col("lon").last(),
    pl.col("lat").last(),
)
distance_ends_expr = distance_ends_expr.over([*index_columns, "jet ID"])

for index_column in index_columns:
    try:
        all_contours = all_contours.cast({index_column: df[index_column].dtype})
    except ColumnNotFoundError:
        pass
all_contours = join_on_ds(all_contours, df)
all_contours = compute_alignment(all_contours, x_periodic)
for indexer, contour in all_contours.group_by([*index_columns, "contour"]):
    lo, la, s, al = contour[["lon", "lat", "s", "alignment"]].to_numpy().T
    clu.axes[2].scatter(
        lo - clu.central_longitude,
        la,
        c=al,
        s=s,
        cmap=BLUEWHITERED,
        vmin=-1.0,
        vmax=1.0,
        edgecolors="black",
    )
valids = (
    explode_rle(
        do_rle(
            all_contours.with_columns(condition=condition_expr),
            [*index_columns, "contour"],
        )
        .with_columns(value=pl.col("value").fill_null(False))
        .filter(
            (pl.col("value").not_() & (pl.col("len") < hole_size)) | pl.col("value")
        )
        .filter(
            pl.col("value").cum_sum().over(*index_columns, "contour") > 0,
            ~(
                ~pl.col("value")
                & (
                    pl.col("start")
                    == pl.col("start").max().over(*index_columns, "contour")
                )
            ),
        )
    )
    .drop("value", "len", "start")
    .group_by([*index_columns, "contour"], maintain_order=True)
    .agg(pl.col("index"), len=pl.col("index").len())
    .filter(pl.col("len") > 5)
    .explode("index")
)
jets = valids.join(all_contours, on=[*index_columns, "contour", "index"])

jets = (
    sort_by_index_then_difflon(
        jets.with_columns(len=pl.len().over([*index_columns, "contour"])).filter(
            pl.col("len") > 6
        ),
        index_columns,
        "contour",
    )
    .with_columns(diff=diff_exp().over([*index_columns, "contour"]))
    .with_columns(
        contour=(pl.col("contour") + 0.01 * (pl.col("diff") > 10).cum_sum())
        .rle_id()
        .over(index_columns)
    )
    .rename({"contour": "jet ID"})
    .drop("cyclic", "len", "len_right")
)
jets = (
    sort_by_index_then_newindex(
        jets.with_columns(len=pl.len().over([*index_columns, "jet ID"])).filter(
            pl.col("len") > 6
        ),
        index_columns,
        "jet ID",
    )
    .with_columns(
        int=int_expr,
        distance_ends=distance_ends_expr,
    )
    .filter(condition_expr2, pl.col("distance_ends") > 1e6)
    .drop("distance_ends")
    .with_columns(pl.col("jet ID").rle_id().over([*index_columns]))
)
jets = sort_by_index_then_newindex(jets, index_columns, "jet ID")
meanlats = jets.group_by("time", "jet ID").agg(pl.col("lat").mean())
order = meanlats["lat"].arg_sort().to_numpy()
for indexer, jet in jets.group_by([*index_columns, "jet ID"], maintain_order=True):
    lo, la, int_ = jet[["lon", "lat", "int"]].to_numpy().T
    int_ = int_[0]
    # int_thresh = int_thresh[0]
    # if int_ < int_thresh:
    #     continue
    if len(lo) < 15:
        continue
    clu.axes[3].scatter(
        lo - clu.central_longitude,
        la,
        color=COLORS[(order[indexer[1]] + 1)],
        label=f"${num2tex(int_, ncomma=2)}$",
    )
# clu.axes[3].legend(ncol=3, framealpha=0.99)

clu.fig.suptitle(f"{np.datetime_as_string(time_to_plot, unit='m')}", fontsize=25)
clu.resize_relative([1.3, 1.1])
clu.fig.savefig(f"{FIGURES}/FeatureBased/detection_demo.pdf")

## bias and bias-correction demo

In [ ]:
time_to_plot = np.random.choice(times)

ds = wind_sample_.sel(time=[time_to_plot]).copy()
n_coarsen: int = 3
smooth_func = partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.7)
ds, ds_orig = preprocess_ds(ds.copy(), n_coarsen=n_coarsen, smooth_func=smooth_func)

clu = Clusterplot(1, 1, get_region(wind_sample), row_height=6)
clu.add_contourf(ds_orig["s"], cmap=colormaps.gray_r, levels=9, transparify=2, cbar_label=r"Wind speed [$\mathrm{m\cdot s}^{-1}$]")

lo, la = ds_orig.lon.values, ds_orig.lat.values
clu.axes[0].contour(lo - clu.central_longitude, la, ds_orig["sigma"][0].values, colors=COLORS[2], levels=[0.], linewidths=1.2)
lo, la = ds.lon.values, ds.lat.values
clu.axes[0].contour(lo - clu.central_longitude, la, ds["sigma"][0].values, colors=COLORS[1], levels=[0.], linewidths=3)
clu.axes[0].set_title(f"{np.datetime_as_string(time_to_plot, unit='m')}")
clu.fig.savefig(f"{FIGURES}/FeatureBased/bias_realspace.pdf")

In [ ]:
path_coarse = Path(RESULTS, "coarse")
jets_coarse = pl.read_parquet(path_coarse.joinpath("jets.parquet"))
bc = pl.read_parquet(path_coarse.joinpath("bias_correct.parquet"))
s_interp = create_jet_relative_dataset(jets_coarse, wind_sample["s"].rename("s")).drop("is_polar")
s_interp_xr = polars_to_xarray(s_interp, ["time", "jet ID", "n", "norm_index"])
s_interp_bc = create_jet_relative_dataset(jets_coarse, wind_sample["s"].rename("s"), bias_correction=bc).drop("is_polar")
s_interp_bc_xr = polars_to_xarray(s_interp_bc, ["time", "jet ID", "n", "norm_index"])

In [ ]:
from matplotlib.colors import Normalize
fig, axes = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True, sharex="all", sharey="all")
tau = s_interp_xr["norm_index"].values
n = s_interp_xr["n"].values / 1e6
norm = Normalize(20, 55)
im = axes[0].pcolormesh(tau, n, s_interp_xr.mean(["time", "jet ID"]).values, cmap=colormaps.gothic_r, norm=norm)
axes[0].axhline(lw=2, color="white")
axes[0].contour(tau, n, s_interp_xr.differentiate("n").mean(["time", "jet ID"]).values, levels=[0.], colors=[COLORS[3]], linewidths=3)
axes[1].pcolormesh(tau, n, s_interp_bc_xr.mean(["time", "jet ID"]).values, cmap=colormaps.gothic_r, norm=norm)
axes[1].axhline(lw=2, color="white")
axes[1].contour(tau, n, s_interp_bc_xr.differentiate("n").mean(["time", "jet ID"]).values, levels=[0.], colors=[COLORS[3]], linewidths=3)
axes[0].set_title("Without bias correction")
axes[1].set_title("With bias correction")
fig.colorbar(im, ax=axes, label=r"Wind speed [$\mathrm{m\cdot s}^{-1}$]", pad=0.01)

for ax in axes:
    ax.set_xlabel(r"$\tau$")
axes[0].set_ylabel(r"$n$ [$10^6$m]")
fig.savefig(f"{FIGURES}/FeatureBased/bias_jetcentred.pdf")

# Testing new jets

In [ ]:
wind_sample = xr.open_dataset(f"{DATADIR}/ERA5/sample_high.nc")
wind_sample = extract(
    wind_sample, minlon=-80, maxlon=40, minlat=15, maxlat=80
)
wind_sample["sigma"] = compute_norm_derivative(wind_sample, "s")
wind_sample_ = wind_sample.compute()

In [ ]:
times = wind_sample["time"].values
path = Path(RESULTS, "test")

find_jets_kwargs = dict(
    n_coarsen=3,
    base_s_thresh=0.55,
    alignment_thresh=0.6,
    int_thresh_factor=0.6,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)

jets_coarse, _ , _, props_coarse = do_everything(wind_sample_, path, **find_jets_kwargs)
bc = pl.read_parquet(path.joinpath("bias_correct.parquet"))

In [ ]:
jets_old = pl.read_parquet(f"{DATADIR}/exp8/all_jets_one_df.parquet").filter(pl.col("time").dt.year() == 1959)

In [ ]:
for t, jets_old_ in jets_old.group_by("time"):
    jets_new_ = jets_coarse.filter(pl.col("time") == t[0])
    plt.scatter(*jets_new_["lon", "lat"])
    plt.scatter(*jets_old_["lon", "lat"], color="red")
    break

In [ ]:
xys = []
all_jets = pl.read_parquet(f"{DATADIR}/results/coarse_bc/jets.parquet")
# .filter(pl.col("int") > 1.1e8)

xys = np.array(xys)
fig, axes = plt.subplots(
    3, 4, figsize=(11, 9), constrained_layout=True, sharex="all", sharey="all"
)
axes = axes.ravel()
pair = ["s", "theta", "is_polar"]
cmap = LinearSegmentedColormap.from_list(
    "pinkredpurple", [COLORS[2], COLORS_EXT[8], COLORS[1]]
)
bins = np.linspace(0, 1, 31)
gridsize = 18
for month in range(1, 13):
    ax = axes[month - 1]
    X = extract_features(all_jets, pair, season=month)
    probas = X[pair[2]]
    center_stj = X.filter(pl.col("is_polar") < 0.3).mean()
    center_edj = X.filter(pl.col("is_polar") > 0.7).mean()
    X1D = X["is_polar"]

    im1 = ax.hexbin(X[pair[0]], X[pair[1]], cmap=colormaps.gray_r, gridsize=gridsize)
    im2 = ax.hexbin(
        X[pair[0]], X[pair[1]], C=probas, cmap=colormaps.gray_r, gridsize=gridsize
    )

    plt.draw()

    offsets1 = np.asarray(list(map(tuple, im1.get_offsets())), dtype="f, f")
    offsets2 = np.asarray(list(map(tuple, im2.get_offsets())), dtype="f, f")
    mask12 = np.isin(offsets1, offsets2)
    colors = cmap(im2.get_array())
    colors = rgb_to_hsv(colors[:, :3])
    min_s, max_s = 0.0, 0.9
    min_v, max_v = 0.9, 1.0
    scaling = np.clip(
        im1.get_array()[mask12] / im1.get_array()[mask12].max() * 1.1, 0, 1
    )
    f = lambda x: np.sqrt(x)
    colors[:, 1] = min_s + scaling * (max_s - min_s)
    colors[:, 2] = max_v - scaling * (max_v - min_v)
    colors = hsv_to_rgb(colors)
    im2.set_array(None)
    im2.set_facecolor(colors)
    # im2.set_linewidths(0.2)
    im2.set_linewidths(np.clip(2 - 3 * scaling, 0, 2))
    im2.set_edgecolor("white")
    im2 = ax.add_collection(im2)
    if month > 8:
        ax.set_xlabel(pair[0])
    if month % 4 == 0:
        ax.set_ylabel(pair[1])
    if pair[0] in ["lev", "theta"]:
        ax.invert_xaxis()
    elif pair[1] in ["lev", "theta"]:
        ax.invert_yaxis()

    ax.set_title(MONTH_NAMES[month - 1])
    ax.scatter(
        *pl.concat([center_stj, center_edj])[pair[:2]].to_numpy().T,
        facecolor="black",
        edgecolor="white",
        marker="X",
        linewidths=1,
        s=45,
    )
    iax = ax.inset_axes([0.65, 0.14, 0.5, 0.5])
    X1D = np.clip(X1D, 0, 1)
    iax.hist(X1D, bins=bins, alpha=0.5, color="black")
    iax.hist(X1D[probas > 0.5], bins=bins, alpha=1.0, color=COLORS[1])
    iax.hist(X1D[probas < 0.5], bins=bins, alpha=1.0, color=COLORS[2])
    iax.set_xticks([0, 0.5, 1])
    iax.set_yticks([])
    iax.spines[["left", "right", "top"]].set_visible(False)
    iax.set_facecolor((1.0, 1.0, 1.0, 0.0))
    plt.draw()
# fig.savefig(f"{FIGURES}/FeatureBased/gmix_demo.pdf")

In [ ]:
data_vars = [
    "mean_lat",
    "s_star",
    "pers",
    "wavinessDC16",
    "njets",
    "double_jet_index",
]

for huh in ["coarse", "coarse_bc", "coarse_bc_s", "fine"]:
    path = Path(RESULTS, huh)
    jets = pl.read_parquet(path.joinpath("s_interp.parquet"))
    huh = polars_to_xarray(jets, ["time", "jet ID", "n", "norm_index"])
    plt.figure()
    huh["s_interp"].differentiate("n").mean("time")[0].plot.contour(levels=[0.])
    plt.gca().axhline(0.)
    # .plot()
    # print(jets.unique(["time", "jet ID"]).shape)

In [ ]:
index_columns = get_index_columns(jets)
orig_jets = jets.clone()
idxmax = jets.select(*index_columns, idxmax=pl.col("index").max().over(index_columns))
if "sigma" not in ds:
    ds["sigma"] = compute_norm_derivative(ds, "s")
jets = gather_normal_da_jets(
    jets, ds["sigma"].compute(), half_length=5e5, dn=2.5e4
)
useful = jets.filter(pl.col("n") == 0)[*index_columns, "index", "lon", "lat", "angle"]
jets = polars_to_xarray(jets, [*index_columns, "index", "n"])[
    "sigma_interp"
]
jets = jets.rolling(index=11, min_periods=1).mean().rolling(n=3, min_periods=1).mean()
jets = (
    detect_contours(
        jets,
        levels=[0.0],
        spatial_dims=("n", "index"),
        do_round=False
    )
    .with_columns(pl.col("index").round().cast(pl.UInt32()))
    .drop_nulls("contour")
    .filter(~pl.col("cyclic"))
    .drop("cyclic")
    .join(idxmax, on=index_columns, how="left")
    .unique([*index_columns, "contour", "index"])
    .sort(*index_columns, "contour", "index")
    .filter(pl.len().over("time", "jet ID", "contour") == pl.len().over("time", "jet ID", "contour").max().over("time", "jet ID"))
    .join(useful, on=[*index_columns, "index"])
)

arc_distances = pl.col("n").first() / RADIUS
lon = pl.col("lon").first().radians()
lat = pl.col("lat").first().radians()
angle = pl.col("angle").first()

newlat = (
    lat.sin() * arc_distances.cos() + lat.cos() * arc_distances.sin() * angle.sin()
).arcsin()
newlat = newlat.degrees()

newlon = lon + pl.arctan2(
    angle.cos() * arc_distances.sin() * lat.cos(),
    arc_distances.cos() - lat.sin() * newlat.sin(),
)
newlon = newlon.degrees()

jets = jets.group_by("time", "jet ID", "index", maintain_order=True).agg(
    lon=newlon,
    lat=newlat,
)
left_behind = orig_jets
print(len(jets))

In [ ]:
col = "angle"
cuts = np.linspace(-0.5, 0.5, 31)
labels = (cuts[:-1] + cuts[1:]) / 2
labels = [f"{cuts[0]:.2f}"] + [f"{l:.2f}" for l in labels] + [f"{cuts[-1]:.2f}"]

for bc in [bias_fine, bias_coarse]:
    bc = bc.with_columns(
        pl.col(col).cut(cuts, labels=labels).cast(pl.String()).cast(pl.Float32())
    )
    bc = bc.group_by(col).agg(pl.col("n").mean()).sort(col)
    plt.plot(bc["angle"], bc["n"])
plt.gca().set_ylim(-5e5, 5e5)
plt.grid(True)

In [ ]:
col = "dangle"
cuts = np.linspace(-0.3, 0.3, 31)
labels = (cuts[:-1] + cuts[1:]) / 2
labels = [f"{cuts[0]:.2f}"] + [f"{l:.2f}" for l in labels] + [f"{cuts[-1]:.2f}"]

for bc in [bias_fine, bias_coarse]:
    bc = bc.with_columns(
        pl.col(col).cut(cuts, labels=labels).cast(pl.String()).cast(pl.Float32())
    )
    bc = bc.group_by(col).agg(pl.col("n").mean()).sort(col)
    plt.plot(bc[col], bc["n"])
plt.gca().set_ylim(-5e5, 5e5)
plt.grid(True)

In [ ]:
join_on_ds(all_contours, ds_orig)

In [ ]:
from jetutils.geospatial import nearest_mapping
ds = wind_sample_.sel(time=[time_to_plot]).copy()
all_contours = detect_contours_lonlat(ds["sigma"], levels=[0.], do_round=False, repeat_lons=0)
index_columns = get_index_columns(all_contours, ["member", "number", "cluster", "time"])
other_indexer = get_index_columns(all_contours, ["jet", "jet ID", "contour", "level"])
if isinstance(ds, xr.DataArray | xr.Dataset):
    ds = xarray_to_polars(ds)
lo = nearest_mapping(all_contours, ds, "lon")
la = nearest_mapping(all_contours, ds, "lat")

(
    all_contours
    .join(lo, on="lon")
    .drop("lon")
    .rename({"lon_": "lon"})
    .join(la, on="lat")
    .drop("lat")
    .rename({"lat_": "lat"})
    .unique([*index_columns, *other_indexer, "lon", "lat"], maintain_order=True)
    .join(ds, on=[*index_columns, "lon", "lat"], how="left")
)

In [ ]:
from jetutils.definitions import _indexer_from_da,_da_to_polars

In [ ]:
_da_to_polars(wind_sample_.sel(time=[time_to_plot])["s"])

In [ ]:
ds = xarray_to_polars(wind_sample_.sel(time=[time_to_plot]))

In [ ]:
ds.filter(pl.col("lon") == -38.0, pl.col("lat") == 15.0)

In [ ]:
wind_sample_.sel(time=[time_to_plot]).sel(lon=-38.0, lat=15.0).load()

In [ ]:
all_contours

In [ ]:
all_contours = detect_contours_lonlat(ds["sigma"], levels=[0.], do_round=False)

In [ ]:
plt.hist(compute_alignment(join_on_ds(detect_contours_lonlat(ds["sigma"], levels=[0.], do_round=False), ds))["alignment"])

In [ ]:
detect_contours_lonlat(ds["sigma"], levels=[0.], do_round=False, repeat_lons=0)

In [ ]:
from tqdm import trange

ds = wind_sample.isel(time=0)
angles = np.atan2(ds["v"], ds["u"]).values
data = ds["s"].values
radius = 50
sigma_tangent, sigma_normal = 10, 3


def directional_smooth(
    ds,
    of: str,
    sigma_tangent: float = 2.0,
    sigma_normal: float = 2.0,
    radius: int | None = None,
):
    if radius is None:
        radius = int(4 * max(sigma_tangent, sigma_normal))
    if radius % 2 == 0:
        radius = radius - 1
    x = np.arange(radius) - radius // 2
    pad = radius // 2
    y = x.copy()
    X = x[None, None, :, None]
    Y = y[None, None, None, :]

    angles = np.atan2(ds["v"], ds["u"]).values
    angles_ = angles[:, :, None, None]
    a = np.cos(angles_) ** 2 / (2 * sigma_tangent**2) + np.sin(angles_) ** 2 / (
        2 * sigma_normal**2
    )
    b = np.sin(2 * angles_) / (4 * sigma_tangent**2) - np.sin(2 * angles_) / (
        4 * sigma_normal**2
    )
    c = np.sin(angles_) ** 2 / (2 * sigma_tangent**2) + np.cos(angles_) ** 2 / (
        2 * sigma_normal**2
    )
    g = (a * X**2) + (2 * b * X * Y) + (c * Y**2)  # parentheses for readability
    g = np.exp(-g) / (2 * np.pi * sigma_tangent * sigma_normal)

    data = ds[of]
    data_ = np.pad(data, (pad, pad))
    data_ = np.lib.stride_tricks.as_strided(
        data_, (radius, radius, *data.shape), strides=data_.strides * 2
    )
    data_ = np.einsum("ijkl,ijkl->kl", data_, g.transpose(3, 2, 0, 1)[::-1, ::-1, ...])
    return ds[of].copy(data=data_)


wind_sample = wind_sample.load()
for i in trange(wind_sample.time.shape[0]):
    directional_smooth(wind_sample.isel(time=0), "s", 10, 3)

In [ ]:
wind_sample

In [ ]:
data_ = np.lib.stride_tricks.as_strided(np.pad(data, (, 4)), (7, 7, *data.shape), strides=data.strides * 2)


In [ ]:
def conv2d(a, f):
    s = f.shape + tuple(np.subtract(a.shape, f.shape) + 1)
    print(f.shape, a.shape, s)
    strd = np.lib.stride_tricks.as_strided
    subM = strd(a, shape=s, strides=a.strides * 2)
    return np.einsum("ij,ijkl->kl", f, subM)


conv2d(orig_sigma[0].values, g[5, 5])

In [ ]:
s = g[5].shape + tuple(np.subtract(orig_sigma[0].values.shape, g[5].shape) + 1)
np.lib.stride_tricks.as_strided(
    orig_sigma[0].values, shape=s, strides=orig_sigma[0].values.strides * 2
).shape

In [ ]:
orig_sigma[0].plot.contour(levels=[0.0], colors="red")
wind_sample["sigma"][0].plot.contour(levels=[0.0], colors="cyan")

In [ ]:
wind_sample["s"][0].plot()
orig_s[0].plot.contour()

In [ ]:
from tqdm import trange

angle = np.atan2(wind_sample["v"], wind_sample["u"])
for i in trange(wind_sample.time.shape[0]):
    conv_with_angle(wind_sample["s"][i].values, 3, 10, angle[i].values)

In [ ]:
ds["s"][0].plot.contourf(levels=[40, 50, 60], cmap=colormaps.greys)
ds["sigma"][0].plot.contour(levels=[0.0], colors="cyan")
ds["s"][0].copy(data=conv_with_angle(data, 3, 10, angles)).plot.contour(levels=[0.0])

In [ ]:
ds["s"][0].plot()

In [ ]:
ds["s"][0].copy(data=conv_with_angle(ds["s"][0].values, 3, 10, angles)).plot()

In [ ]:
plt.plot(np.arange(7) - 3)
plt.grid(True)

In [ ]:
conv_with_angle(data, sigma_tangent, sigma_normal, angles):
    

In [ ]:
from skimage.filters import sato, hessian

nx, ny = 3, 3
clu = Clusterplot(ny, nx, region=get_region(wind_sample), row_height=4)
time_to_plot = np.random.choice(times, nx * ny)
ds = wind_sample.sel(time=time_to_plot).copy()
s_orig = ds["s"].copy()

with Timer():
    ds["sato"] = ds["s"].copy(
        data=[
            hessian(ds["s"][i], sigmas=range(8, 20, 2), black_ridges=True)
            for i in range(len(ds["s"]))
        ]
    )
# ds["s"] = ds["u"].copy(data=gaussian_filter(ds["u"], (3, 8), axes=(1, 2)))
# ds["u"] = ds["u"].copy(data=gaussian_filter(ds["u"], (3, 8), axes=(1, 2)))
# ds["v"] = ds["v"].copy(data=gaussian_filter(ds["v"], (3, 8), axes=(1, 2)))

ds["ssigma"] = compute_norm_derivative(ds, "sato")
sigma = compute_norm_derivative(ds, "s")

sigma = gaussian_filter(sigma, (2, 10), axes=(1, 2))
ds["sigma"] = ds["s"].copy(data=sigma)

_ = clu.add_contourf(ds["sato"], cmap=colormaps.greys, levels=15, transparify=2)
# lo, la = ds.lon.values, ds.lat.values
for i in range(len(time_to_plot)):
    clu.axes[i].contour(
        lo - clu.central_longitude,
        la,
        ds["sigma"][i],
        levels=[0.0],
        linewidths=2,
        colors="cyan",
    )
    clu.axes[i].contour(
        lo - clu.central_longitude,
        la,
        ds["ssigma"][i],
        levels=[0.0],
        linewidths=2,
        colors="red",
    )

# gmix

In [ ]:
rgb_to_hsv(hex2color(COLORS[1]))

In [ ]:
from matplotlib.colors import hex2color

rgb_to_hsv(hex2color(COLORS[2]))

In [ ]:
xys = []
all_jets = pl.read_parquet(f"{DATADIR}/exp8/all_jets_one_df.parquet")
# .filter(pl.col("int") > 1.1e8)

xys = np.array(xys)
fig, axes = plt.subplots(
    3, 4, figsize=(11, 9), constrained_layout=True, sharex="all", sharey="all"
)
axes = axes.ravel()
pair = ["s", "theta", "is_polar"]
cmap = LinearSegmentedColormap.from_list(
    "pinkredpurple", [COLORS[2], COLORS_EXT[8], COLORS[1]]
)
bins = np.linspace(0, 1, 31)
for month in range(1, 13):
    ax = axes[month - 1]
    X = extract_features(all_jets, pair, season=month)
    probas = X[pair[2]]
    center_stj = X.filter(pl.col("is_polar") < 0.3).mean()
    center_edj = X.filter(pl.col("is_polar") > 0.7).mean()
    X1D = X["is_polar"]

    im1 = ax.hexbin(X[pair[0]], X[pair[1]], cmap=colormaps.gray_r, gridsize=25)
    im2 = ax.hexbin(
        X[pair[0]], X[pair[1]], C=probas, cmap=colormaps.gray_r, gridsize=25
    )

    plt.draw()

    offsets1 = np.asarray(list(map(tuple, im1.get_offsets())), dtype="f, f")
    offsets2 = np.asarray(list(map(tuple, im2.get_offsets())), dtype="f, f")
    mask12 = np.isin(offsets1, offsets2)
    colors = cmap(im2.get_array())
    colors = rgb_to_hsv(colors[:, :3])
    min_s, max_s = 0.0, 0.9
    min_v, max_v = 0.9, 1.0
    scaling = np.clip(
        im1.get_array()[mask12] / im1.get_array()[mask12].max() * 1.1, 0, 1
    )
    f = lambda x: np.sqrt(x)
    colors[:, 1] = min_s + scaling * (max_s - min_s)
    colors[:, 2] = max_v - scaling * (max_v - min_v)
    colors = hsv_to_rgb(colors)
    im2.set_array(None)
    im2.set_facecolor(colors)
    # im2.set_linewidths(0.2)
    im2.set_linewidths(np.clip(2 - 3 * scaling, 0, 2))
    im2.set_edgecolor("white")
    im2 = ax.add_collection(im2)
    if month > 8:
        ax.set_xlabel(pair[0])
    if month % 4 == 0:
        ax.set_ylabel(pair[1])
    if pair[0] in ["lev", "theta"]:
        ax.invert_xaxis()
    elif pair[1] in ["lev", "theta"]:
        ax.invert_yaxis()

    ax.set_title(MONTH_NAMES[month - 1])
    ax.scatter(
        *pl.concat([center_stj, center_edj])[pair[:2]].to_numpy().T,
        facecolor="black",
        edgecolor="white",
        marker="X",
        linewidths=1,
        s=45,
    )
    iax = ax.inset_axes([0.65, 0.14, 0.5, 0.5])
    X1D = np.clip(X1D, 0, 1)
    iax.hist(X1D, bins=bins, alpha=0.5, color="black")
    iax.hist(X1D[probas > 0.5], bins=bins, alpha=1.0, color=COLORS[1])
    iax.hist(X1D[probas < 0.5], bins=bins, alpha=1.0, color=COLORS[2])
    iax.set_xticks([0, 0.5, 1])
    iax.set_yticks([])
    iax.spines[["left", "right", "top"]].set_visible(False)
    iax.set_facecolor((1.0, 1.0, 1.0, 0.0))
    plt.draw()
fig.savefig(f"{FIGURES}/FeatureBased/gmix_demo.pdf")

## twostep

In [ ]:
centers_kmeans = xr.open_dataarray(
    "/Users/bandelol/Documents/code_local/data/exp3_kmeans/centers_kmeans_7_200.nc"
)
labels_kmeans = xr.open_dataarray(
    "/Users/bandelol/Documents/code_local/data/exp3_kmeans/labels_kmeans_7_200.nc"
)
time = labels_kmeans.time

clu = Clusterplot(4, 2, row_height=4, region=get_region(centers))
_ = clu.add_contourf(
    centers,
    cmap=colormaps.gothic_r,
    levels=[0, 12, 24, 36, 48],
    cbar_kwargs={"label": "Wind speed [m/s]", "pad": 0.01},
)
clu.fig.savefig(
    "/Users/bandelol/Documents/code_local/local_figs/twostep_kmeans_centers.png"
)
plt.show()
clear_output()

weekly = (
    xarray_to_polars(labels_kmeans)
    .group_by(pl.col("time").dt.week().alias("week"))
    .agg(pl.col("labels").unique(), pl.col("labels").unique_counts().alias("count"))
    .explode("labels", "count")
    .sort("week", "labels")
)
unique_weeks = weekly["week"].unique().to_frame()
unique_labels = weekly["labels"].unique().to_frame()
index = unique_weeks.join(unique_labels, how="cross")
weekly = index.join(weekly, on=index.columns, how="left").fill_null(0)

bottom = np.zeros(len(unique_weeks))
x = unique_weeks["week"].to_numpy()
cmap_ = colormaps.pastel
colors = cmap_(np.linspace(0, 1, cmap_.N))
fig, ax = plt.subplots()
for lab in unique_labels["labels"].to_numpy():
    y = weekly.filter(pl.col("labels") == lab)["count"]
    plt.bar(x, y, bottom=bottom, color=colors[lab])
    bottom = bottom + y
ax.set_ylim(0, bottom.max() * 1.04)
clu.fig.savefig(
    "/Users/bandelol/Documents/code_local/local_figs/twostep_kmeans_pops.png"
)


In [ ]:
for i in range(3, 8):
    labels_kmeans = load_pickle(
        f"/Users/bandelol/Documents/code_local/data/exp3_kmeans/k_{i}_200.pkl"
    ).labels_

    weekly = (
        pl.DataFrame(
            [pl.Series("time", time.values), pl.Series("labels", labels_kmeans)]
        )
        .group_by(pl.col("time").dt.week().alias("week"))
        .agg(pl.col("labels").unique(), pl.col("labels").unique_counts().alias("count"))
        .explode("labels", "count")
        .sort("week", "labels")
    )
    unique_weeks = weekly["week"].unique().to_frame()
    unique_labels = weekly["labels"].unique().to_frame()
    index = unique_weeks.join(unique_labels, how="cross")
    weekly = index.join(weekly, on=index.columns, how="left").fill_null(0)

    bottom = np.zeros(len(unique_weeks))
    x = unique_weeks["week"].to_numpy()
    cmap_ = colormaps.pastel
    colors = cmap_(np.linspace(0, 1, cmap_.N))
    fig, ax = plt.subplots()
    for lab in unique_labels["labels"].to_numpy():
        y = weekly.filter(pl.col("labels") == lab)["count"]
        plt.bar(x, y, bottom=bottom, color=colors[lab])
        bottom = bottom + y
    ax.set_ylim(0, bottom.max() * 1.04)
    ax.set_title(f"Kmeans, $k={i}$")
    clu.fig.savefig(
        f"/Users/bandelol/Documents/code_local/local_figs/twostep_kmeans_pops_{i}.png"
    )